# U-Net — Convolutional Networks for Biomedical Image Segmentation (2015)

_Papers · Computer Vision_

**A U-shaped encoder-decoder with skip connections turns a classifier into a per-pixel segmenter — and trains well from very few images.**

---

This notebook is a practice scaffold for the **U-Net — Convolutional Networks for Biomedical Image Segmentation (2015)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(0)

# --- 0. Sanity-check the worked example: spatial size through the U (valid convs, paper's 572). ---
def conv2(n): return n - 4          # two valid 3x3 convs subtract 2 each
def pool(n):  return n // 2         # 2x2 stride-2 max-pool halves
def up(n):    return n * 2          # 2x2 up-conv doubles
n = 572; down = [n]
for _ in range(4): n = pool(conv2(n)); down.append(n)   # 4 encoder blocks + pools
n = conv2(n); down.append(n)                            # bottleneck
print("contracting sizes:", [conv2(572), pool(conv2(572))], "... bottleneck =", n)  # 568, 284, ..., 28
for _ in range(4): n = conv2(up(n))                     # 4 decoder blocks
print("final output size :", n, "(paper Fig. 1: 388)")  # 388


# --- 1. A TOY segmentation task we generate: 32x32 images with a random thin ring; mask = ring pixels. ---
H = 32
def make_batch(num):
    yy, xx = torch.meshgrid(torch.arange(H), torch.arange(H), indexing="ij")
    imgs = torch.zeros(num, 1, H, H); masks = torch.zeros(num, 1, H, H)
    for i in range(num):
        cy = torch.randint(7, H - 7, (1,)).item(); cx = torch.randint(7, H - 7, (1,)).item()
        r  = torch.randint(4, 7, (1,)).item()
        d2 = (yy - cy) ** 2 + (xx - cx) ** 2
        ring = (d2 <= r * r) & (d2 >= (r - 2) * (r - 2))     # thin outline -> needs fine localization
        masks[i, 0] = ring.float()
        imgs[i, 0]  = ring.float() * 0.9 + torch.randn(H, H) * 0.25   # noisy ring
    return imgs, masks


# --- 2. The U-Net. use_skip toggles the concatenating skip connections (the ablation switch). ---
def double_conv(ci, co):                                # two 3x3 convs (padded) + ReLU each
    return nn.Sequential(nn.Conv2d(ci, co, 3, padding=1), nn.ReLU(),
                         nn.Conv2d(co, co, 3, padding=1), nn.ReLU())

class UNet(nn.Module):
    def __init__(self, use_skip=True):
        super().__init__(); self.use_skip = use_skip
        self.enc1 = double_conv(1, 16)                  # contracting path
        self.enc2 = double_conv(16, 32)
        self.pool = nn.MaxPool2d(2)
        self.bott = double_conv(32, 64)                 # bottleneck
        self.up2  = nn.ConvTranspose2d(64, 32, 2, stride=2)          # expanding path (up-convs)
        self.dec2 = double_conv(64 if use_skip else 32, 32)         # 64 = 32 up-conv + 32 skip
        self.up1  = nn.ConvTranspose2d(32, 16, 2, stride=2)
        self.dec1 = double_conv(32 if use_skip else 16, 16)         # 32 = 16 up-conv + 16 skip
        self.outc = nn.Conv2d(16, 1, 1)                 # 1x1 head -> one logit per pixel
    def forward(self, x):
        e1 = self.enc1(x)                               # 16 x 32 x 32
        e2 = self.enc2(self.pool(e1))                   # 32 x 16 x 16
        b  = self.bott(self.pool(e2))                   # 64 x  8 x  8  (bottleneck)
        d2 = self.up2(b)                                # 32 x 16 x 16
        if self.use_skip: d2 = torch.cat([d2, e2], 1)   # SKIP: concatenate encoder features
        d2 = self.dec2(d2)
        d1 = self.up1(d2)                               # 16 x 32 x 32
        if self.use_skip: d1 = torch.cat([d1, e1], 1)   # SKIP
        return self.outc(self.dec1(d1))                 # 1 x 32 x 32 logits


def dice(logits, mask):                                 # overlap metric (ignores easy background)
    p = (logits.sigmoid() > 0.5).float()
    inter = (p * mask).sum((1, 2, 3))
    return ((2 * inter) / (p.sum((1, 2, 3)) + mask.sum((1, 2, 3)) + 1e-6)).mean().item()

def train(use_skip, steps=600):
    torch.manual_seed(0)
    net = UNet(use_skip=use_skip); opt = torch.optim.Adam(net.parameters(), lr=1e-3)
    for s in range(steps):
        x, m = make_batch(16)
        loss = F.binary_cross_entropy_with_logits(net(x), m)   # per-pixel CE (Eq. 1, w==1 here)
        opt.zero_grad(); loss.backward(); opt.step()
        if s % 150 == 0: print(f"  step {s:3d}  loss {loss.item():.4f}")
    return net


def show_mask(t, title):                                # print a predicted mask as ASCII
    print(title); g = t[0, 0, 8:24, 8:24]
    for row in g: print("".join("#" if v > 0.5 else "." for v in row))


print("TRAIN full U-Net (with skips):")
net = train(use_skip=True)
torch.manual_seed(99); xt, mt = make_batch(64)
with torch.no_grad(): lt = net(xt)
print("test Dice (with skips):", round(dice(lt, mt), 3))     # our run: ~0.99

# --- 3. Show predicted masks on one held-out image. ---
torch.manual_seed(7); xi, mi = make_batch(1)
with torch.no_grad(): pa = net(xi).sigmoid()
show_mask(mi, "\nGROUND TRUTH ring:")
show_mask(pa, "\nU-NET prediction (with skips) -- crisp:")

# --- 4. ABLATION: remove the concatenating skips, retrain identically, compare. ---
print("\nTRAIN ablated U-Net (NO skips):")
net_no = train(use_skip=False)
with torch.no_grad():
    lt2 = net_no(xt); pb = net_no(xi).sigmoid()
print("test Dice (NO skips):", round(dice(lt2, mt), 3))      # our run: ~0.92  -> worse
show_mask(pb, "\nNO-SKIP prediction -- blurrier / broken edges:")
# With skips ~0.99 Dice and a clean ring; without, the ring thickens and breaks.
# (Our small run -- not the paper's reported number.)

## Visualize the data & results

_On a toy ring-segmentation task, do the concatenating skip connections give sharper predicted masks than the same encoder-decoder without them?_

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(0)

H = 32
def make_batch(num):
    yy, xx = torch.meshgrid(torch.arange(H), torch.arange(H), indexing="ij")
    imgs = torch.zeros(num, 1, H, H); masks = torch.zeros(num, 1, H, H)
    for i in range(num):
        cy = torch.randint(7, H - 7, (1,)).item(); cx = torch.randint(7, H - 7, (1,)).item()
        r  = torch.randint(4, 7, (1,)).item()
        d2 = (yy - cy) ** 2 + (xx - cx) ** 2
        ring = (d2 <= r * r) & (d2 >= (r - 2) * (r - 2))
        masks[i, 0] = ring.float()
        imgs[i, 0]  = ring.float() * 0.9 + torch.randn(H, H) * 0.25
    return imgs, masks

def double_conv(ci, co):
    return nn.Sequential(nn.Conv2d(ci, co, 3, padding=1), nn.ReLU(),
                         nn.Conv2d(co, co, 3, padding=1), nn.ReLU())

class UNet(nn.Module):
    def __init__(self, use_skip=True):
        super().__init__(); self.use_skip = use_skip
        self.enc1 = double_conv(1, 16); self.enc2 = double_conv(16, 32); self.pool = nn.MaxPool2d(2)
        self.bott = double_conv(32, 64)
        self.up2 = nn.ConvTranspose2d(64, 32, 2, stride=2); self.dec2 = double_conv(64 if use_skip else 32, 32)
        self.up1 = nn.ConvTranspose2d(32, 16, 2, stride=2); self.dec1 = double_conv(32 if use_skip else 16, 16)
        self.outc = nn.Conv2d(16, 1, 1)
    def forward(self, x):
        e1 = self.enc1(x); e2 = self.enc2(self.pool(e1)); b = self.bott(self.pool(e2))
        d2 = self.up2(b)
        if self.use_skip: d2 = torch.cat([d2, e2], 1)
        d2 = self.dec2(d2); d1 = self.up1(d2)
        if self.use_skip: d1 = torch.cat([d1, e1], 1)
        return self.outc(self.dec1(d1))

def dice(lg, m):
    p = (lg.sigmoid() > 0.5).float(); inter = (p * m).sum((1, 2, 3))
    return ((2 * inter) / (p.sum((1, 2, 3)) + m.sum((1, 2, 3)) + 1e-6)).mean().item()

def train(use_skip, steps=600):
    torch.manual_seed(0); net = UNet(use_skip=use_skip); opt = torch.optim.Adam(net.parameters(), 1e-3)
    for _ in range(steps):
        x, m = make_batch(16)
        loss = F.binary_cross_entropy_with_logits(net(x), m)
        opt.zero_grad(); loss.backward(); opt.step()
    return net

torch.manual_seed(99); xt, mt = make_batch(64)
for use_skip in (True, False):
    net = train(use_skip)
    with torch.no_grad(): d = dice(net(xt), mt)
    print(("with skips " if use_skip else "no  skips  "), "Dice =", round(d, 3))
# with skips Dice = 0.991 ; no skips Dice = 0.921  -> skips give the crisp masks.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The ablation. Your full U-Net segments the toy rings almost perfectly. Now remove the
            skip connections (delete the two torch.cat calls so the decoder gets only the
            up-convolved maps) and retrain everything else identically. What happens to the predicted
            masks, and what does it prove?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Set use_skip=False so each decoder level feeds forward only the up-convolved map &mdash; no encoder features concatenated. Keep channels, depth, optimizer, data, and step count identical. — _An honest ablation changes exactly one thing &mdash; the skip connections &mdash; so any difference in the masks is attributable to them alone._
- Retrain and look at the predicted ring. It comes out thicker, broken, and ragged at the edges, and the held-out Dice overlap drops noticeably. — _Without the skips the only route to the output runs through the pooled bottleneck, which discarded the pixel-precise edge locations; the up-convolutions can only produce a smooth, low-resolution guess._
- Conclude that the skip connections, not the encoder-decoder shape by itself, are what deliver crisp boundaries. — _Both networks have the same context-capturing encoder; only the one with skips can put the fine detail back, which is the paper's central claim._

**Answer:** The masks get blurrier and broken &mdash; the ring thickens, develops gaps, and
                 loses its clean edge &mdash; and the Dice overlap falls (in our run from ~0.99 with skips
                 to ~0.92 without). Because the skips are the only thing removed, this isolates them as the
                 source of precise localization: the encoder-decoder shape gives context, but it is
                 the concatenated encoder features that restore sharp boundaries. (The CODE includes this
                 ablation.)

</details>

**Problem 2.** Trace the spatial size for a smaller input. With valid 3&times;3 convolutions, you feed a
            $64\times64$ patch through one encoder block (two convs), one 2&times;2 pool, and one more
            block (two convs). What is the size after each stage?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Block 1 (two valid 3&times;3 convs): $64 \to 62 \to 60$, i.e. $64-4 = 60$. — _Each valid 3&times;3 conv subtracts 2 from each side; two of them subtract 4._
- Pool (2&times;2, stride 2): $60 \to 30$. — _Max-pool with stride 2 halves height and width._
- Block 2 (two valid convs): $30 \to 26$, i.e. $30-4 = 26$. — _Same conv rule again._

**Answer:** $64 \xrightarrow{\text{2 convs}} 60 \xrightarrow{\text{pool}} 30
                 \xrightarrow{\text{2 convs}} 26$. The map at the deeper level is $26\times26$; to skip the
                 block-1 output ($60\times60$) across to a later same-level decoder map you would
                 center-crop it to the decoder's size &mdash; exactly the cropping the paper's gray arrows
                 perform.

</details>

**Problem 3.** Why does U-Net concatenate the encoder feature map onto the decoder rather than
            add it (the way a residual/ResNet skip does)?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Note what each operation requires: addition needs the two tensors to have the same number of channels and assumes they are directly comparable; concatenation just stacks them, allowing different channel counts and meanings. — _The encoder map (fine, early features) and the decoder map (coarse, deep features) represent different things; forcing them to add would entangle them._
- Observe that after concatenation the next 3&times;3 conv sees both maps as separate input channels and learns a weighting to combine them. — _The network decides how much to trust the fine encoder stream vs. the coarse decoder stream, per feature &mdash; more expressive than a fixed sum._
- Recall the cost: concatenation doubles the channel count entering that conv (hence the 64-channel input from a 32+32 join), so the conv is wider. — _That extra width is the price for keeping both signals intact &mdash; the paper accepts it for the localization gain._

**Answer:** Concatenation keeps the fine encoder features and the coarse decoder features as
                 separate channels, so the following convolution can learn how to merge them
                 (and they need not even have the same channel count). Addition would force them into one
                 blended signal of equal width. The cost is a wider conv (doubled input channels), which
                 U-Net pays to preserve both context and precise detail.

</details>